In [ ]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.linear_model import LassoCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

## 1. Load clean data

In [ ]:
# Load clean data
X_test_student = pd.read_csv('X_test_student.csv', index_col=0)
X_train_student = pd.read_csv('X_train_student.csv', index_col=0)
X_test_worker = pd.read_csv('X_test_worker.csv', index_col=0)
X_train_worker = pd.read_csv('X_train_worker.csv', index_col=0)

y_test_student = pd.read_csv('y_test_student.csv').squeeze()
y_train_student = pd.read_csv('y_train_student.csv').squeeze()
y_test_worker = pd.read_csv('y_test_worker.csv').squeeze()
y_train_worker = pd.read_csv('y_train_worker.csv').squeeze()

print(f'Dataset shape: {X_train_student.shape}')
print(f'\nColumns: {X_train_student.columns.tolist()}')
print(f'\nData types:\n{X_train_student.dtypes}')
print(f'\nMissing values:\n{X_train_student.isnull().sum()}')
print(f'\nFirst few rows:')
X_train_student.head()

In [ ]:
X_train_worker.head()

In [ ]:
binary_cols = ['Depression', 
               'Have you ever had suicidal thoughts ?',
               'Family History of Mental Illness']

ordinal_cols = ['Sleep Duration', 'Dietary Habits']

#numeric_cols = ['Age', 'CGPA', 'Work/Study Hours']

categorical_cols = ['Gender', 'Profession', 'Degree']

high_card_cols = ['City']

## 2.  Categorical feature encoding

Categorical columns:

- Gender (Male / Female / Other): 3 distinct categories, no meaningful ordinal relationship

- Profession (10 unique categories): multi-class nominal feature with no natural hierarchical structure.

- Degree (7 unique categories): although educational levels may appear ordered, in this dataset the categories do not follow a strict or consistent hierarchy.

=> Using OneHotEncoder

- Sleep Duration (3 ordered categories): 'Less than or equal 6 hours' < '6-8 hours' < 'More than 8 hours'

- Dietary Habits (3 ordered categories): 'Unhealthy' < 'Moderate'  < 'Healthy'

=> Using OrdinalEncoder

- City (high cardinality categorical feature)

=> Using Frequency encoding

In [ ]:

from sklearn.base import BaseEstimator, TransformerMixin

class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.freq_maps = {}


    def fit(self, X, y=None):
        for col in X.columns:
            self.freq_maps[col] = X[col].value_counts(normalize=True).to_dict()
        return self

    def transform(self, X):
        X_transformed = X.copy()
        for col in X_transformed.columns:
            X_transformed[col] = X_transformed[col].map(self.freq_maps[col]).fillna(0)
        return X_transformed

In [ ]:
def encode_categorical_features(X_train, X_test, categorical_cols, high_card_cols, ordinal_cols):

    categorical_cols = [
        col for col in categorical_cols 
        if X_train[col].nunique() > 1
    ]   
    
    ordinal_maps = [
    ['Less than or equal 6 hours', '6-8 hours', 'More than 8 hours'],  # Sleep Duration
    ['Unhealthy', 'Moderate', 'Healthy']   # Dietary Habits
    ]  
    ordinal_pipeline = Pipeline([ 
        ('ordinal', OrdinalEncoder(categories=ordinal_maps)) 
    ])
    
    categorical_pipeline = Pipeline([
        ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
    ])

    high_card_pipeline = Pipeline([
        ('freq', FrequencyEncoder())
    ])

    preprocessor = ColumnTransformer([
        ('ordinal', ordinal_pipeline, ordinal_cols),
        ('low_card', categorical_pipeline, categorical_cols),
        ('high_card', high_card_pipeline, high_card_cols)
    ], remainder='passthrough')

    X_train_encoded = preprocessor.fit_transform(X_train)
    X_test_encoded = preprocessor.transform(X_test)

    ordinal_names = ordinal_cols 
    onehot_names = preprocessor.named_transformers_['low_card']['onehot'].get_feature_names_out(categorical_cols) 
    passthrough_cols = [col for col in X_train.columns if col not in ordinal_cols + categorical_cols + high_card_cols] 
    feature_names = list(ordinal_names) + list(onehot_names) + list(high_card_cols) + passthrough_cols
    
    X_train_new = pd.DataFrame(X_train_encoded, columns=feature_names, index=X_train.index)
    X_test_new = pd.DataFrame(X_test_encoded, columns=feature_names, index=X_test.index)
    
    return X_train_new, X_test_new

In [ ]:
X_train_student, X_test_student = encode_categorical_features(X_train_student, X_test_student, categorical_cols, high_card_cols, ordinal_cols)
X_train_student.head()

With student dataset drop columns: Name, Profession

In [ ]:
columns_to_drop = ["Name", "Profession"]
X_train_student.drop(columns_to_drop, axis=1, inplace=True)
X_test_student.drop(columns_to_drop, axis=1, inplace=True)

In [ ]:
X_train_worker, X_test_worker = encode_categorical_features(X_train_worker, X_test_worker, categorical_cols, high_card_cols, ordinal_cols)
X_train_worker.head()

With dataset for worker drop columns: Name

In [ ]:
columns_to_drop = ["Name"]
X_train_worker.drop(columns_to_drop, axis=1, inplace=True)
X_test_worker.drop(columns_to_drop, axis=1, inplace=True)

In [ ]:
def fix_numeric_types(df):
    df = df.copy()
    numeric_cols = df.columns  # ?p t?t c?, column n?o kh?ng th? ?p s? th?nh NaN
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
    return df


## 3. Create new features and interactions

### 3.1. Create new features and interactions for student data

- Burnout_Load = Academic Pressure ? Work/Study Hours -> ?p l?c h?c l?n + nhi?u gi? h?c -> overload, nh?ng ?p l?c h?c l?n + ?t gi? h?c -> thi?u ??ng l?c -> stress do y?u t? kh?c 

- Effort_Reward_Imbalance = Burnout_load x (6 - Study Satisfaction) -> nguy c? tr?m c?m t?ng n?u nh? high effort, low satisfaction

- Performance_Anxiety_Score = CGPA x Academic Pressure -> CGPA th?p m? ?p l?c cao ? depression risk r?t l?n, CGPA cao m? ?p l?c cao ? nh?m ?perfectionist? ? also risky

### 3.2. Create new features and interactions for worker data

Burnout_Load = Work Pressure ? Work/Study Hours: ?p l?c c?ng vi?c + nhi?u gi? l?m -> nguy c? overload

Effort_Reward_Imbalance = Burnout_load x (6 - Job Satisfaction) -> nguy c? tr?m c?m t?ng n?u nh? high effort, low satisfaction

Trapped_Score = Financial Stress + (6 - Job Satisfaction) -> v?n ?? t?i ch?nh + ch?n c?ng vi?c -> kh?ng th? ngh? vi?c, m? v?n k?t ??. 

### 3.3. Create new features and interactions for all

- Physical Health = Sleep Hours x Dietarity Habits -> Ng? ?? gi?c, ?n u?ng l?nh m?nh/ ng? thi?u gi? + unhealthy: s?c kh?e th? ch?t c? ?nh h??ng t?i s?c kh?e tinh th?n ko?

- Work_Sleep = Work/Study Hours x Sleep Hours -> ng? ?t h?c nhi?u d?n t?i burning out, tr?m c?m

- Suicidal Thoughts x Family Mental History -> suy ngh? t? t? + ti?n s? gia ??nh c? g?y ra tr?m c?m 

- Sleep_Vulnerability = Financial Stress x Sleep Hours -> c?ng th?ng t?i ch?nh -> lo l?ng, ng? ?t

- Financial Stress x Work/ Study Hours: stress t?i ch?nh-> l?m/ nhi?u gi? h?n ?? sau c? t??ng lai t?t...

- Age_Pressure = Pressure / Age -> ng??i tr? th??ng d? b? ?p l?c h?n so v?i l?n tu?i h?n

In [ ]:
# 4.1. Create new features and interactions for student data
def create_student_features(df):
    df = df.copy()
    # Burnout_Load = Academic Pressure * Work/Study Hours
    df['Burnout_Load'] = df['Academic Pressure'] * df['Work/Study Hours']
    
    # Effort_Reward_Imbalance = Burnout_Load * (6 - Study Satisfaction)
    df['Effort_Reward_Imbalance'] = df['Burnout_Load'] * (6 - df['Study Satisfaction'])
    
    # Performance_Anxiety_Score = CGPA * Academic Pressure
    df['Performance_Anxiety_Score'] = df['CGPA'] * df['Academic Pressure']
    
    return df

# 4.2. Create new features and interactions for worker data
def create_worker_features(df):
    df = df.copy()
    # Burnout_Load = Work Pressure * Work/Study Hours
    df['Burnout_Load'] = df['Work Pressure'] * df['Work/Study Hours']
    
    # Effort_Reward_Imbalance = Burnout_Load * (6 - Job Satisfaction)
    df['Effort_Reward_Imbalance'] = df['Burnout_Load'] * (6 - df['Job Satisfaction'])
    
    # Trapped_Score = Financial Stress + (6 - Job Satisfaction)
    df['Trapped_Score'] = df['Financial Stress'] + (6 - df['Job Satisfaction'])
    
    return df

# 4.3. Create new features and interactions for all
def create_common_features(df, is_student=True):
    df = df.copy()
    # Physical Health = Sleep Duration * Dietary Habits
    df['Physical_Health'] = df['Sleep Duration'] * df['Dietary Habits']
    
    # Work_Sleep = Work/Study Hours * Sleep Duration
    df['Work_Sleep'] = df['Work/Study Hours'] * df['Sleep Duration']
    
    # Suicidal Thoughts * Family Mental History
    df['Suicidal_Family_Interaction'] = df['Have you ever had suicidal thoughts ?'] * df['Family History of Mental Illness']
    
    # Sleep_Vulnerability = Financial Stress * Sleep Duration
    df['Sleep_Vulnerability'] = df['Financial Stress'] * df['Sleep Duration']
    
    # Financial Stress * Work/ Study Hours
    df['Financial_Work_Interaction'] = df['Financial Stress'] * df['Work/Study Hours']
    
    # Age_Pressure = Pressure / Age
    # Adding 5 to Age to avoid division by zero and sign flipping issues near mean
    if is_student:
        df['Age_Pressure'] = df['Academic Pressure'] / (df['Age'] + 5)
    else:
        df['Age_Pressure'] = df['Work Pressure'] / (df['Age'] + 5)
        
    return df

# Apply features
X_train_student = fix_numeric_types(X_train_student)
X_test_student = fix_numeric_types(X_test_student)

X_train_student = create_student_features(X_train_student)
X_train_student = create_common_features(X_train_student, is_student=True)

X_test_student = create_student_features(X_test_student)
X_test_student = create_common_features(X_test_student, is_student=True)

X_train_worker = fix_numeric_types(X_train_worker)
X_test_worker  = fix_numeric_types(X_test_worker)

X_train_worker = create_worker_features(X_train_worker)
X_train_worker = create_common_features(X_train_worker, is_student=False)

X_test_worker = create_worker_features(X_test_worker)
X_test_worker = create_common_features(X_test_worker, is_student=False)

print("Features created successfully.")


## 4. Numeric features scaling

Scale numerical columns using StandardScaler 

In [ ]:
numeric_cols_student = X_train_student.select_dtypes(include=['int64','float64']).columns.tolist()
numeric_cols_worker  = X_train_worker.select_dtypes(include=['int64','float64']).columns.tolist()

In [ ]:
print("Student numeric:", numeric_cols_student)
print("Worker numeric:", numeric_cols_worker)


In [ ]:
def scale_data(X_train, X_test, numeric_cols):
    scaler = StandardScaler()

    X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
    X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])
    return X_train, X_test

In [ ]:
X_train_student, X_test_student = scale_data(
    X_train_student, X_test_student, numeric_cols_student
)

X_train_worker, X_test_worker = scale_data(
    X_train_worker, X_test_worker, numeric_cols_worker
)


In [ ]:
import pandas as pd

# Student
df_student = X_train_student.copy()
df_student['Depression'] = y_train_student.values  # ch?c ch?n c?ng index

# Worker
df_worker = X_train_worker.copy()
df_worker['Depression'] = y_train_worker.values

In [ ]:
X_train_student.head()

## 5. Feature selection

Select features by using:

- Pearson correlation: rank features by absolute linear correlation with target.

- Mutual Information: rank features by non-linear dependency with target.

- VIF: check multicollinearity among features.

- LassoCV: embedded feature selection via regularized linear regression coefficients.

- RandomForest importance: rank features by contribution to tree-based model predictions.

- Aggregate ranking: combine ranks from all methods to get mean rank.

Select top features: choose top 10 features based on aggregated ranking.


In [ ]:
# %pip install statsmodels

In [ ]:
# VIF helper (uses statsmodels if available)
def compute_vif(df, features):
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    X = df[features].assign(constant=1.0)
    vif_data = {f: variance_inflation_factor(X.values, i) for i, f in enumerate(X.columns) if f != 'constant'}
    return pd.Series(vif_data).sort_values(ascending=False)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def feature_selection(X_train, y_train):
    # Pearson correlation with target (absolute)
    corrs = X_train.corrwith(y_train).abs().sort_values(ascending=False)
    print('\nTop 15 features by abs(correlation) with target:')
    print(corrs.head(15))

    # Mutual Information (captures non-linear relationships)
    mi = mutual_info_classif(X_train.fillna(0), y_train)
    mi_series = pd.Series(mi, index=X_train.columns).sort_values(ascending=False)
    print('\nTop 15 features by mutual information:')
    print(mi_series.head(15))

    # VIF (optional - requires statsmodels)
    vif = compute_vif(X_train.fillna(0), X_train.columns.tolist())
    if vif is not None:
        print('\nTop VIF values:')
        print(vif.sort_values(ascending=True).head(15))

    # LassoCV for embedded selection
    lasso = LassoCV(cv=5, random_state=42, n_jobs=-1)
    lasso.fit(X_train.fillna(0), y_train)
    coef_abs = pd.Series(np.abs(lasso.coef_), index=X_train.columns).sort_values(ascending=False)
    print('\nTop 15 features by absolute Lasso coefficients:')
    print(coef_abs.head(15))

    # RandomForest importance
    rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    rf.fit(X_train.fillna(0), y_train)
    rf_imp = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
    print('\nTop 15 features by RandomForest importance:')
    print(rf_imp.head(15))

    # Aggregate a simple ranking (average rank across methods)
    rank_df = pd.DataFrame({
        'corr_rank': corrs.rank(ascending=False),
        'mi_rank': mi_series.rank(ascending=False),
        'lasso_rank': coef_abs.rank(ascending=False),
        'rf_rank': rf_imp.rank(ascending=False)
    }).fillna(1e6) # missing treated as low importance

    rank_df['mean_rank'] = rank_df.mean(axis=1)
    ranked = rank_df.sort_values('mean_rank')

    print('\nTop 20 features by aggregated mean rank:')
    print(ranked.head(20))

    # Save selected top features
    selected_features = ranked.head(10).index.tolist()
    print('\nSelected features (top 10):', selected_features)
    return selected_features


In [ ]:
student_features = feature_selection(X_train_student, y_train_student)

In [ ]:
worker_features = feature_selection(X_train_worker, y_train_worker)

In [ ]:
# This code is temporary
X_train_student = X_train_student[student_features]
X_test_student = X_test_student[student_features]
X_train_worker = X_train_worker[worker_features]
X_test_worker = X_test_worker[worker_features]

student_model = RandomForestClassifier(random_state=42)
worker_model = RandomForestClassifier(random_state=42)

student_model.fit(X_train_student, y_train_student)
worker_model.fit(X_train_worker, y_train_worker)
y_pred_student = student_model.predict(X_test_student)
y_pred_worker = worker_model.predict(X_test_worker)

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

print("Classification Report for Student MODEL")
print(classification_report(y_test_student, y_pred_student))

# V? ma tr?n
ConfusionMatrixDisplay.from_predictions(y_test_student, y_pred_student, display_labels=["Normal", "Depressed"], cmap="Blues")
plt.show()


In [ ]:
print("Classification Report for Worker MODEL")
print(classification_report(y_test_worker, y_pred_worker))

# V? ma tr?n
ConfusionMatrixDisplay.from_predictions(y_test_worker, y_pred_worker, display_labels=["Normal", "Depressed"], cmap="Blues")
plt.show()

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=f6527928-bbf4-4c9a-8760-1f30ce0e5ba3' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>